In [1]:
import pandas as pd
import feedparser
import urllib
import time
import calendar
import re

# Getting the GW Detections data from LVK

We use the ``.csv`` file from the LVK website, which contains all the detections and their properties. We will use this data to get the number of detections per month, which we will then compare to the number of astrophysics papers published per month.

In [2]:
# Make Observation runs dataframe
## Note: Currenlty only O1 - 04a event data has been published, but the rest of the O4 runs are included for future use when the data is released.
runs = pd.DataFrame({
    "Run": ["O1", "O2", "O3a", "O3b", "O4a", "O4b", "O4c1", "O4c2"],
    "Start Date": ["2015-09-12", "2016-11-30", "2019-04-01", "2019-11-01", "2023-05-24", "2024-04-10", "2025-01-28", "2025-06-12"],
    "End Date": ["2016-01-19", "2017-08-25", "2019-09-30", "2020-03-27", "2024-01-16", "2025-01-28", "2025-03-31", "2025-11-18"],
    "Detectors": [2, 3, 3, 3, 3, 4, 4, 4]
})
runs["Start Date"] = pd.to_datetime(runs["Start Date"])
runs["End Date"] = pd.to_datetime(runs["End Date"])
runs["Detectors"] = runs["Detectors"].astype(int)

runs

,Run,Start Date,End Date,Detectors
0,O1,2015-09-12,2016-01-19,2
1,O2,2016-11-30,2017-08-25,3
2,O3a,2019-04-01,2019-09-30,3
3,O3b,2019-11-01,2020-03-27,3
4,O4a,2023-05-24,2024-01-16,3
5,O4b,2024-04-10,2025-01-28,4
6,O4c1,2025-01-28,2025-03-31,4
7,O4c2,2025-06-12,2025-11-18,4


In [3]:
# Reading the event data
gw = pd.read_csv(r"../01 - Data/raw/events.csv")
gw.head()

# Adding a detection date column: based on the GW event naming convention, the date is the first part of the name, in the format "GWYYYYMMDD". We will extract this date and convert it to a datetime object.
gw["Detection Date"] = pd.to_datetime(gw["name"].str.split("_").str[0].str.replace("GW", "20"), format="%Y%m%d")
gw.head()

,name,shortName,gps,version,catalog,doi,detail_url,mass_1_source,mass_1_source_lower,mass_1_source_upper,...,far,far_lower,far_upper,p_astro,p_astro_lower,p_astro_upper,final_mass_source,final_mass_source_lower,final_mass_source_upper,Detection Date
0,GW240109_050431,GW240109_050431-v1,1.388812e+09,1,GWTC-4.0,https://doi.org/10.7935/aes8-px89,https://gwosc.org/api/v2/event-versions/GW2401...,28.8,-6.2,7.5,...,0.00023,NaN,NaN,0.99,NaN,NaN,45.1,-5.5,6.5,2024-01-09
1,GW240107_013215,GW240107_013215-v1,1.388626e+09,1,GWTC-4.0,https://doi.org/10.7935/aes8-px89,https://gwosc.org/api/v2/event-versions/GW2401...,59.0,-18.0,27.0,...,0.02800,NaN,NaN,0.99,NaN,NaN,87.0,-19.0,28.0,2024-01-07
2,GW240105_151143,GW240105_151143-v1,1.388503e+09,1,GWTC-4.0,https://doi.org/10.7935/aes8-px89,https://gwosc.org/api/v2/event-versions/GW2401...,NaN,NaN,NaN,...,3.30000,NaN,NaN,0.70,NaN,NaN,NaN,NaN,NaN,2024-01-05
3,GW240104_164932,GW240104_164932-v1,1.388422e+09,1,GWTC-4.0,https://doi.org/10.7935/aes8-px89,https://gwosc.org/api/v2/event-versions/GW2401...,42.3,-6.7,9.4,...,0.00001,NaN,NaN,0.99,NaN,NaN,70.6,-7.9,10.7,2024-01-04
4,GW231231_154016,GW231231_154016-v1,1.388072e+09,1,GWTC-4.0,https://doi.org/10.7935/aes8-px89,https://gwosc.org/api/v2/event-versions/GW2312...,22.5,-3.3,5.7,...,0.00001,NaN,NaN,0.99,NaN,NaN,38.1,-3.2,4.0,2023-12-31


In [4]:
# Assigning the observation run to each event
def assign_run(date):
    for _, row in runs.iterrows():
        if row["Start Date"] <= date <= row["End Date"]:
            return row["Run"]
    return None

# Cleaned event data with assigned observation runs. We only keep the relevant columns for our analysis: event name, detection date, observation run, year, and month. The year and month columns will be useful for time series analysis and visualizations later on.
gw_clean = pd.DataFrame({
    "Event": gw["name"],
    "Detection Date": gw["Detection Date"],
    "Observation Run": gw["Detection Date"].apply(assign_run),
    "Year": gw["Detection Date"].dt.to_period("Y").dt.to_timestamp(),
    "Month": gw["Detection Date"].dt.to_period("M").dt.to_timestamp()
})

# Merge the event data with the observation runs to get the number of detectors for each event: useful for later analysis of how the number of detectors affects the number of detections.
gw_clean = gw_clean.merge(
    runs[["Run", "Detectors"]],
    left_on="Observation Run",
    right_on="Run",
    how="left"
)

gw_clean.drop(columns=["Run"], inplace=True)

gw_clean.info()
gw_clean.head()


<class 'pandas.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Event            219 non-null    str           
 1   Detection Date   219 non-null    datetime64[us]
 2   Observation Run  218 non-null    str           
 3   Year             219 non-null    datetime64[us]
 4   Month            219 non-null    datetime64[us]
 5   Detectors        218 non-null    float64       
dtypes: datetime64[us](3), float64(1), str(2)
memory usage: 10.4 KB


,Event,Detection Date,Observation Run,Year,Month,Detectors
0,GW240109_050431,2024-01-09,O4a,2024-01-01,2024-01-01,3.0
1,GW240107_013215,2024-01-07,O4a,2024-01-01,2024-01-01,3.0
2,GW240105_151143,2024-01-05,O4a,2024-01-01,2024-01-01,3.0
3,GW240104_164932,2024-01-04,O4a,2024-01-01,2024-01-01,3.0
4,GW231231_154016,2023-12-31,O4a,2023-01-01,2023-12-01,3.0


In [5]:
# Making a new dataframe for montlhy analysis
gw_monthly = gw_clean.groupby(["Month"]).agg({"Event": "count", "Detectors": "mean"}).reset_index()
gw_monthly.rename(columns={"Event": "Event Count", "Detectors": "Avg Detectors"}, inplace=True)

# Create full monthly date range, to avoid avoid missing months with no detections
full_range = pd.date_range(
    start=gw_monthly["Month"].min(),  # Starting at the start of the first detection to capture all months.
    end=gw_monthly["Month"].max(),  # Ending after the end of the last detection to capture all months.
    freq="MS"  # Month Start
)

# Reindex the monthly dataframe to include all months, filling missing months with NaN values for event count and average detectors.
gw_monthly = (gw_monthly.set_index("Month")
              .reindex(full_range)
              .rename_axis("Month")
              .reset_index()
)

# Fill missing values: for event count, we fill missing months with 0, while for average detectors, we forward fill the last known value, as the number of detectors does not change until a new observation run starts.
gw_monthly["Event Count"] = gw_monthly["Event Count"].fillna(0)
gw_monthly["Avg Detectors"] = gw_monthly["Avg Detectors"].ffill()

# Calculate cumulative detections and time index for plotting: the cumulative detections will help us visualize the overall growth in detections over time, while the time index will be useful for plotting the data on a timeline and timeseries analysis.
gw_monthly["Cumulative Detections"] = gw_monthly["Event Count"].cumsum()
gw_monthly["Time Index"] = range(len(gw_monthly))

# Setting the correct data types for the columns.
gw_monthly["Event Count"] = gw_monthly["Event Count"].astype(int)
gw_monthly["Avg Detectors"] = gw_monthly["Avg Detectors"].astype(int)
gw_monthly["Cumulative Detections"] = gw_monthly["Cumulative Detections"].astype(int)

gw_monthly.info()
gw_monthly.head()

# Writing the cleaned and processed monthly data to a new CSV file for later use in analysis and visualizations.
gw_monthly.to_csv(r"../01 - Data/processed/gw_monthly.csv", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Month                  101 non-null    datetime64[us]
 1   Event Count            101 non-null    int64         
 2   Avg Detectors          101 non-null    int64         
 3   Cumulative Detections  101 non-null    int64         
 4   Time Index             101 non-null    int64         
dtypes: datetime64[us](1), int64(4)
memory usage: 4.1 KB


# Getting the papers data using the arXiv API

We will use the arXiv API to get the number of papers published per month about gravitational waves. We will use the ``feedparser`` library to parse the arXiv feed and extract the relevant information.

We will also get the monthly count of astrophysics papers in general, to compare the trends in GW papers to the overall trends in astrophysics research.

In [6]:
# Defining the analysis range
start_year = 2010
end_year = 2024 # Ending in 2024 to capture all papers published during and after O4a, which is the latest observation run with published data.

# Although it is higly unlikely, we create a full monthly date range to ensure we don't miss any months with no papers, which could cause issues in later analysis and visualizations.
full_range = pd.date_range(
    start=f"{start_year}-01-01",
    end=f"{end_year}-12-01",
    freq="MS"
    )

In [7]:
# Fetching data from arxiv,possible to get more granular by month.
def fetch_arxiv(query, start_year, end_year, start_month=1, end_month=12):
    all_entries = []
    
    for year in range(start_year, end_year + 1):
        for month in range(start_month, end_month + 1):
            last_day = calendar.monthrange(year, month)[1]
            start = 0
            # Build readable query, following arXiv's syntax for date range filtering
            date_filter = f"submittedDate:[{year}{month:02d}010000 TO {year}{month:02d}{last_day:02d}2359]"
            full_query = f"{query} AND {date_filter}"
            # URL encode entire query
            encoded_query = urllib.parse.quote(full_query)

            # Loop while we reach the last page of results for the current month, fetching 100 results at a time until there are no more entries to fetch.
            while True:
                url = (
                    "https://export.arxiv.org/api/query?"
                    f"search_query={encoded_query}"
                    f"&start={start}"
                    f"&max_results=100"
                )
                feed = feedparser.parse(url)
                entries = feed.entries
                # Just a failsafe to break the loop if there are no entries, which means we've reached the end of the results for the current month.
                if not entries:
                    break
                # Get the relevant data from each entry and append it to the list of all entries. We also clean the ID by removing any version numbers (e.g., "v1", "v2") to ensure we can deduplicate later on.
                for entry in entries:
                    entry_id = getattr(entry, "id", None).split("/")[-1]
                    clean_id = re.sub(r'v\d+$', '', entry_id)
                    published = getattr(entry, "published", None)
                    title = getattr(entry, "title", None)
                    authors = getattr(entry, "authors", None)

                    if published and title and authors:
                        all_entries.append({
                            "id": clean_id,
                            "title": title,
                            "published": published,
                            "total_authors": len(authors)
                        })
                # Stop if last page
                if len(entries) < 100:
                    break
                
                # Increment the start index for the next page of results and sleep briefly to avoid hitting rate limits
                start += 100
                time.sleep(3)
    
    # Make a dataframe from the collected entries
    df = pd.DataFrame(all_entries)
    # Deduplicate by ID
    df = df.drop_duplicates(subset="id")
    
    return df

In [8]:
# Fetching arXiv papers related to gravitational waves
all_years = []

# Get all papers with "gravitational wave" in the title, abstract, or keywords, published between 2010 and 2024. This will capture papers published during the entire period of GW detections, as well as a few years before to capture any foundational papers that may have been published before the first detection in 2015. We fetch year by year to avoid hitting the arXiv API limits and to ensure we capture all relevant papers without missing any due to pagination issues.
for year in range(start_year, end_year + 1):
    print(f"Fetching {year}")
    df_year = fetch_arxiv("all:gravitational+wave", year, year)
    df_year.to_csv(rf"../01 - Data/raw/gw_papers_{year}.csv", index=False)
    print(f"    Fetched {len(df_year)} papers for {year}")
    all_years.append(df_year)

# Combine all years into a single dataframe and deduplicate by ID
gw_papers = pd.concat(all_years, ignore_index=True)
gw_papers = gw_papers.drop_duplicates(subset="id")

gw_papers.info()
gw_papers.head()

Fetching 2010
    Fetched 539 papers for 2010
Fetching 2011
    Fetched 561 papers for 2011
Fetching 2012
    Fetched 540 papers for 2012
Fetching 2013
    Fetched 577 papers for 2013
Fetching 2014
    Fetched 693 papers for 2014
Fetching 2015
    Fetched 691 papers for 2015
Fetching 2016
    Fetched 993 papers for 2016
Fetching 2017
    Fetched 1167 papers for 2017
Fetching 2018
    Fetched 1358 papers for 2018
Fetching 2019
    Fetched 1600 papers for 2019
Fetching 2020
    Fetched 1767 papers for 2020
Fetching 2021
    Fetched 1825 papers for 2021
Fetching 2022
    Fetched 1921 papers for 2022
Fetching 2023
    Fetched 2177 papers for 2023
Fetching 2024
    Fetched 2351 papers for 2024
<class 'pandas.DataFrame'>
RangeIndex: 18760 entries, 0 to 18759
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             18760 non-null  str  
 1   title          18760 non-null  str  
 2   published      18760 non-null  s

,id,title,published,total_authors
0,1001.3951,Gravitomagnetism and gravitational waves,2010-01-22T10:17:38Z,2
1,1001.1555,Propagation of Gravitational Waves in Generali...,2010-01-11T13:41:53Z,1
2,1001.0198,Features of gravitational waves in higher dime...,2010-01-01T00:37:10Z,1
3,1001.4821,Sources and technology for an atomic gravitati...,2010-01-26T23:05:47Z,8
4,1001.5012,Cross-Correlating Probes of Primordial Gravita...,2010-01-27T20:13:02Z,1


In [9]:
# Making monthly dataframe for GW papers
gw_papers["published"] = pd.to_datetime(gw_papers["published"])
gw_papers["Month"] = gw_papers["published"].dt.to_period("M").dt.to_timestamp()

gw_papers_monthly = gw_papers.groupby("Month").agg({"id": "count", "total_authors": "mean"}).reset_index()
gw_papers_monthly.rename(columns={"id": "Paper Count", "total_authors": "Avg Authors"}, inplace=True)

# Reindex the monthly dataframe to include all months, filling missing months with NaN values for paper count and average authors.
gw_papers_monthly = (gw_papers_monthly.set_index("Month")
              .reindex(full_range)
              .rename_axis("Month")
              .reset_index()
)

# Fill missing values: we fill missing months with 0
gw_papers_monthly["Paper Count"] = gw_papers_monthly["Paper Count"].fillna(0)
gw_papers_monthly["Avg Authors"] = gw_papers_monthly["Avg Authors"].fillna(0)

# Making cumulative paper count and time index for plotting and timeseries analysis
gw_papers_monthly["Cumulative Papers"] = gw_papers_monthly["Paper Count"].cumsum()
gw_papers_monthly["Time Index"] = range(len(gw_papers_monthly))

# Setting the correct data types for the columns
gw_papers_monthly["Avg Authors"] = gw_papers_monthly["Avg Authors"].astype(int)

gw_papers_monthly.info()
gw_papers_monthly.head()

gw_papers_monthly.to_csv(r"../01 - Data/processed/gw_papers_monthly.csv", index=False)


<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Month              180 non-null    datetime64[us]
 1   Paper Count        180 non-null    int64         
 2   Avg Authors        180 non-null    int64         
 3   Cumulative Papers  180 non-null    int64         
 4   Time Index         180 non-null    int64         
dtypes: datetime64[us](1), int64(4)
memory usage: 7.2 KB


C:\Users\muham\AppData\Local\Temp\ipykernel_28668\1937247245.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  gw_papers["Month"] = gw_papers["published"].dt.to_period("M").dt.to_timestamp()


In [10]:
# Fetching arXiv papers related to astrophysics
all_years = []

# Get all astrophysics papers published between 2010 and 2024. This will capture papers published during the entire period of GW detections, as well as a few years before. We fetch month by month to avoid hitting the arXiv API limits and to ensure we capture all relevant papers without missing any due to pagination issues.
for year in range(start_year, end_year + 1):
    print(f"Fetching {year}")
    for month in range(1, 13):
        print(f"    Fetching {month:02d}/{year}")
        df_month = fetch_arxiv("cat:astro-ph*", year, year, start_month=month, end_month=month)
        df_month.to_csv(rf"../01 - Data/raw/astro_papers_{year}_{month:02d}.csv", index=False)
        print(f"        Fetched {len(df_month)} papers for {month:02d}/{year}")
        all_years.append(df_month)

# Combine all years into a single dataframe and deduplicate by ID
astro_papers = pd.concat(all_years, ignore_index=True)
astro_papers = astro_papers.drop_duplicates(subset="id")

astro_papers.info()
astro_papers.head()

Fetching 2010
    Fetching 01/2010
        Fetched 1018 papers for 01/2010
    Fetching 02/2010
        Fetched 959 papers for 02/2010
    Fetching 03/2010
        Fetched 1085 papers for 03/2010
    Fetching 04/2010
        Fetched 1020 papers for 04/2010
    Fetching 05/2010
        Fetched 1139 papers for 05/2010
    Fetching 06/2010
        Fetched 1061 papers for 06/2010
    Fetching 07/2010
        Fetched 1005 papers for 07/2010
    Fetching 08/2010
        Fetched 1024 papers for 08/2010
    Fetching 09/2010
        Fetched 1289 papers for 09/2010
    Fetching 10/2010
        Fetched 1190 papers for 10/2010
    Fetching 11/2010
        Fetched 1248 papers for 11/2010
    Fetching 12/2010
        Fetched 1080 papers for 12/2010
Fetching 2011
    Fetching 01/2011
        Fetched 1162 papers for 01/2011
    Fetching 02/2011
        Fetched 1023 papers for 02/2011
    Fetching 03/2011
        Fetched 1121 papers for 03/2011
    Fetching 04/2011
        Fetched 1019 papers for 04/20

,id,title,published,total_authors
0,1001.4941,The First Release of the CSTAR Point Source Ca...,2010-01-27T13:38:26Z,32
1,1001.4980,The diffuse radio filament in the merging syst...,2010-01-27T16:16:16Z,5
2,1001.5021,Active galaxy unification in the era of X-ray ...,2010-01-27T20:49:11Z,2
3,1001.5038,Scaled oscillation frequencies and echelle dia...,2010-01-27T21:02:55Z,2
4,1001.5042,The host galaxies of core-collapse supernovae ...,2010-01-27T21:25:00Z,5


In [11]:
# Making monthly dataframe for astrophysics papers
astro_papers["published"] = pd.to_datetime(astro_papers["published"])
astro_papers["Month"] = astro_papers["published"].dt.to_period("M").dt.to_timestamp()

astro_papers_monthly = astro_papers.groupby("Month").agg({"id": "count", "total_authors": "mean"}).reset_index()
astro_papers_monthly.rename(columns={"id": "Paper Count", "total_authors": "Avg Authors"}, inplace=True)

# Reindex the monthly dataframe to include all months, filling missing months with NaN values for paper count and average authors.
astro_papers_monthly = (astro_papers_monthly.set_index("Month")
              .reindex(full_range)
              .rename_axis("Month")
              .reset_index()
)

# Fill missing values: we fill missing months with 0
astro_papers_monthly["Paper Count"] = astro_papers_monthly["Paper Count"].fillna(0)
astro_papers_monthly["Avg Authors"] = astro_papers_monthly["Avg Authors"].fillna(0)

# Making cumulative paper count and time index for plotting and timeseries analysis
astro_papers_monthly["Cumulative Papers"] = astro_papers_monthly["Paper Count"].cumsum()
astro_papers_monthly["Time Index"] = range(len(astro_papers_monthly))

# Setting the correct data types for the columns
astro_papers_monthly["Avg Authors"] = astro_papers_monthly["Avg Authors"].astype(int)

astro_papers_monthly.info()
astro_papers_monthly.head()

astro_papers_monthly.to_csv(r"../01 - Data/processed/astro_papers_monthly.csv", index=False)


<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Month              180 non-null    datetime64[us]
 1   Paper Count        180 non-null    int64         
 2   Avg Authors        180 non-null    int64         
 3   Cumulative Papers  180 non-null    int64         
 4   Time Index         180 non-null    int64         
dtypes: datetime64[us](1), int64(4)
memory usage: 7.2 KB


C:\Users\muham\AppData\Local\Temp\ipykernel_28668\681934403.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  astro_papers["Month"] = astro_papers["published"].dt.to_period("M").dt.to_timestamp()
